# P139 — Un modelo de tipos y niveles de interacción humana con la automatización

## 1. Título y paper

**Paper:** *A model for types and levels of human interaction with automation*  
**Autoría:** Raja Parasuraman, Thomas B. Sheridan, Christopher D. Wickens  
**Año y venue:** 2000 · IEEE Transactions on Systems, Man, and Cybernetics, 30(3), 286–297  
**Nivel:** L2 · **Motor:** `niveles_de_automatizacion`  
**Ficha completa:** [`P139_niveles_de_automatizacion`](../../papers/foundational/P139_niveles_de_automatizacion/README.md)

**Hito:** Descompone la automatización en cuatro etapas con diez niveles cada una, y documenta que subir de nivel deja al humano fuera del bucle justo cuando más falta hace.

- [doi:10.1109/3468.844354](https://doi.org/10.1109/3468.844354)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: «Automatizar» se trataba como una decisión de todo o nada sobre un sistema entero. Y subir el nivel tiene un coste que nadie contabilizaba: quien deja de revisar pierde la práctica que le permitía detectar el fallo cuando ocurre.
2. Ejecutar una implementación mínima de la propuesta: Separar cuatro etapas —adquirir información, analizarla, decidir la acción y ejecutarla— y elegir el nivel de automatización de cada una por separado, evaluando el efecto sobre la carga mental, la conciencia de la situación y la confianza del operador.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P134


## 4. Intuición

Automatizar la aprobación no elimina los errores del sistema: elimina a quien los veía. Y quien revisa menos, además revisa **peor** — sin práctica no hay criterio.


## 5. Concepto mínimo

```text
Cuatro etapas, cada una con su nivel:
  adquirir información · analizarla · decidir la acción · ejecutarla

nivel 1  : el humano decide todo
nivel 10 : el sistema actúa e ignora al humano
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('niveles_de_automatizacion', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos errores detecta el nivel más bajo? ¿Y el más alto?
2. ¿Cae la detección en proporción a la revisión?
3. ¿Qué sí cae limpiamente?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('niveles_de_automatizacion', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('niveles_de_automatizacion', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con la **misma** tanda de 150 errores para todos los niveles: el nivel 1 detecta **141** y el nivel 10, **0**. Y la caída es superlineal — del nivel 5 al 7 se revisa 2,75× menos y se detecta **3,4× menos** (62 → 18). Lo que sí cae limpiamente es la carga: de 2 000 revisiones a 0.


## 10. Comentario pedagógico

Esa superlinealidad es la «ironía de la automatización» de Bainbridge: al operador se le deja la tarea de vigilar justo aquello para lo que la automatización le ha quitado la práctica. Y por eso el artículo insiste en elegir el nivel **por etapa**: automatizar la adquisición de información casi nunca tiene coste, y automatizar la decisión casi siempre.


## 11. Error o anti-patrón deliberado

Anti-patrón: medir el éxito de un despliegue por cuántas aprobaciones humanas se eliminaron.


In [ ]:
print('Eliminar aprobaciones baja la carga, que es real y se puede medir.')
print('Y sube los errores que pasan, que tambien es real y casi nunca se mide.')
print('Un despliegue que solo reporta la primera mitad no esta reportando.')

## 12. Corrección

Los seis niveles, con la misma tanda de errores:


In [ ]:
r = run_paper_lab('niveles_de_automatizacion', seed=3)['result']
for f in r['por_nivel']:
    print(f"nivel {f['nivel']:>2} | revisa {f['fraccion_revisada_por_el_humano']:.2f}"
          f" | detecta {f['errores_detectados']:>3} | pasan {f['errores_que_pasan']:>3}"
          f" | carga {f['carga_de_revision']}")

## 13. Desafío guiado

Explica por qué el nivel adecuado depende del coste de un error que pasa, y no solo de la carga que se ahorra.


In [ ]:
r = run_paper_lab('niveles_de_automatizacion', seed=3)['result']
show(r)

## 14. Desafío autónomo

Elige una decisión que tu sistema automatice. Sitúala en los diez niveles y estima qué fracción de errores detectaría un humano en el nivel actual.


## 15. Evidencia de aprendizaje

Guarda el nivel, la estimación y el coste de un error no detectado.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P139_niveles_de_automatizacion/README.md) · evaluación formal: [`assessments/papers/P139_niveles_de_automatizacion.md`](../../assessments/papers/P139_niveles_de_automatizacion.md)


## 16. Cierre

Falta el último ingrediente operativo: repartir trabajo pesado entre muchas máquinas.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
